# Starter: prvsiyan's Extended Name That LLM Dataset

One prompt, every model. This notebook (1) loads the dataset and shows how complete the prompt blocks are, (2) trains a prompt-grouped TF-IDF baseline on the dataset alone, and (3) inside Kaggle, evaluates on the competition's train set: a model trained on this dataset only, and competition-only vs competition + dataset under prompt-grouped folds. The competition files are read from `/kaggle/input` and never copied.

In [ ]:
import os, glob
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from scipy.sparse import hstack

def find_data_dir():
    if os.environ.get('NTL_DATA_DIR'):
        return os.environ['NTL_DATA_DIR']
    hits = glob.glob('/kaggle/input/**/passages_clean.csv', recursive=True)
    return os.path.dirname(hits[0]) if hits else '.'

DATA = find_data_dir()
passages = pd.read_csv(os.path.join(DATA, 'passages.csv'))
clean = pd.read_csv(os.path.join(DATA, 'passages_clean.csv'))
prompts = pd.read_csv(os.path.join(DATA, 'prompts.csv'))
blocks = pd.read_csv(os.path.join(DATA, 'blocks.csv'))
print(f'{len(passages):,} passages, {len(clean):,} clean, {passages.prompt_id.nunique()} prompts, {passages.model.nunique()} models')
print(passages.groupby('split').agg(rows=('id', 'size'), prompts=('prompt_id', 'nunique'), models=('model', 'nunique')))

## Blocks and flags
Every prompt was sent once to every model of its split. `blocks.csv` says which models are present and passing; `label_summary.csv` and `train_vs_dataset_stats.csv` summarise each model (the latter against competition-train aggregates, with known register differences flagged).

In [ ]:
print(blocks.groupby('split')[['is_complete', 'is_complete_clean']].mean().round(3))
flags = ['is_refusal', 'is_truncated', 'has_markdown', 'in_word_range', 'passes_funnel']
passages[passages.split == 'new'].groupby('model')[flags].mean().mul(100).round(1)

In [ ]:
tv = pd.read_csv(os.path.join(DATA, 'train_vs_dataset_stats.csv'))
tv[['label', 'train_median_words', 'dataset_median_words', 'train_emdash_share', 'dataset_emdash_share', 'register_flags']]

## Prompt-grouped TF-IDF baseline on the dataset alone
Folds are grouped by `prompt_id`, so a prompt never appears in both training and validation.

In [ ]:
def feats(train_text, *others):
    vc = TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 4), min_df=2, sublinear_tf=True)
    vw = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b|[^\w\s]")
    Xtr = hstack([vc.fit_transform(train_text), vw.fit_transform(train_text)]).tocsr()
    return [Xtr] + [hstack([vc.transform(t), vw.transform(t)]).tocsr() for t in others]

df = clean[clean.split == 'new'].reset_index(drop=True)
labels = sorted(df.model.unique())
y = df.model.values
oof = np.empty(len(df), dtype=object)
for tr, va in GroupKFold(n_splits=min(5, df.prompt_id.nunique())).split(df, y, df.prompt_id):
    Xtr, Xva = feats(df.text.iloc[tr], df.text.iloc[va])
    oof[va] = LogisticRegression(C=10, max_iter=2000, class_weight='balanced').fit(Xtr, y[tr]).predict(Xva)
print(f'dataset-only prompt-grouped balanced accuracy: {balanced_accuracy_score(y, oof):.3f} (chance {1 / len(labels):.3f})')
pd.Series({l: float((oof[y == l] == l).mean()) for l in labels}).round(3).sort_values()

## Evaluate on the competition train set (inside Kaggle)
Attach the competition **Name That LLM** to the notebook. The competition rows stay in `/kaggle/input` (rules forbid redistributing them). (a) Train on this dataset only and score every competition train passage: none of the `new` / `new2` / `proxy_v1` prompts is a competition prompt. (b) Prompt-grouped 5-fold CV on the competition train set, competition-only vs competition + this dataset (dataset rows are added to training folds, never scored). Rows of split `train_regen` answer reconstructions of the competition's own train prompts, so they join a training fold only when their `source_prompt_id` is in it (otherwise they would leak the held-out prompt).

In [ ]:
comp_path = os.environ.get('NTL_COMP_TRAIN') or next(iter(glob.glob('/kaggle/input/name-that-llm/train.csv') + glob.glob('/kaggle/input/**/name-that-llm/train.csv', recursive=True)), None)
if comp_path and os.path.exists(comp_path):
    comp = pd.read_csv(comp_path)
    ext = clean[clean.split.isin(['new', 'new2', 'proxy_v1'])]
    reg = clean[clean.split == 'train_regen']
    Xe, Xc = feats(ext.text, comp.text)
    pred = LogisticRegression(C=10, max_iter=2000, class_weight='balanced').fit(Xe, ext.model).predict(Xc)
    print('(a) trained on this dataset only, scored on competition train:', round(balanced_accuracy_score(comp.model, pred), 3))
    yc = comp.model.values
    res = {}
    for name, extra, use_reg in [('competition only', ext.iloc[:0], False), ('competition + dataset', ext, True)]:
        oofc = np.empty(len(comp), dtype=object)
        for tr, va in GroupKFold(n_splits=5).split(comp, yc, comp.prompt_id):
            r = reg[reg.source_prompt_id.isin(set(comp.prompt_id.iloc[tr]))] if use_reg else reg.iloc[:0]   # train_regen rows follow their competition prompt's fold
            tr_text = pd.concat([comp.text.iloc[tr], extra.text, r.text]); tr_y = np.concatenate([yc[tr], extra.model.values, r.model.values])
            Xtr, Xva = feats(tr_text, comp.text.iloc[va])
            oofc[va] = LogisticRegression(C=10, max_iter=2000, class_weight='balanced').fit(Xtr, tr_y).predict(Xva)
        res[name] = round(balanced_accuracy_score(yc, oofc), 4)
    print('(b) prompt-grouped CV on competition train:', res)
else:
    print('competition data not attached; skipping (attach the Name That LLM competition to run this cell)')

## Provenance
`provenance.jsonl` holds every API call: request body, raw response, usage, requested and served ids, identity checks. `provenance_summary.json` aggregates it per model.

In [ ]:
import json
summary = json.load(open(os.path.join(DATA, 'provenance_summary.json')))
pd.DataFrame({k: {'passages': v['passages'], 'routes': v['routes'], 'served_ids': v['served_model_ids']} for k, v in summary['labels'].items()}).T